# Demo Chương 5: Phân tích Rò rỉ Mạng qua Mô hình Học máy

**Môn học:** An toàn và Bảo mật Hệ thống Thông tin
**Trọng tâm Demo:** Xây dựng mô hình phân tích lưu lượng mạng để phát hiện Rò rỉ Dữ liệu (Data Leakage)

Trong bài thực hành này, chúng ta tập trung phân tích các luồng kết nối (network flow) nhằm phát hiện kịch bản nội gián (insider threat) - một nhân viên cố tình truyền dữ liệu ra ngoài mạng vượt thẩm quyền ngoài giờ làm việc.

## 1. Sinh dữ liệu giả lập (Data Generation)

Tạo 5,000 bản ghi dữ liệu mạng dạng flow với tỉ lệ 85% bình thường (Normal) và 15% rò rỉ dữ liệu (Leakage).

In [ ]:
import sys; sys.path.append('src')
from generate_data import generate_dataset

df = generate_dataset()
df.head()

## 2. Tiền xử lý dữ liệu (Preprocessing)

Quy trình tiền xử lý bao gồm:
- Làm sạch dữ liệu và loại bỏ các cột không sử dụng (IP, Timestamp).
- Mã hóa các biến phân loại (`protocol`).
- Thay đổi đặc trưng dẫn xuất: `upload_download_ratio = bytes_sent / bytes_recv`.
- Chuẩn hóa các biến liên tục MinMaxScaler (khoảng [0,1]).
- Chia tập huấn luyện/kiểm tra 80:20.

In [ ]:
from preprocess import run_preprocessing_pipeline

X_train, X_test, y_train, y_test, scaler, le = run_preprocessing_pipeline()
print('Training set size:', X_train.shape)
print('Testing set size:', X_test.shape)
X_train.head()

## 3. Huấn luyện Mô hình

- **Logistic Regression (Baseline):** Mô hình phân loại tuyến tính cơ bản.
- **Random Forest (Chính):** Mô hình Decision Tree Ensemble.
- **Isolation Forest:** Mô hình phát hiện bất thường Unsupervised (Không giám sát - chỉ sử dụng nhãn Normal khi train).

In [ ]:
from train_models import train_all_models

models = train_all_models(X_train, y_train)

## 4. Đánh giá Kết quả (Evaluation)

Sử dụng các chỉ số: **Accuracy**, **Precision**, **Recall**, **F1-Score**, **FPR** và biểu đồ **Confusion Matrix**.

In [ ]:
from evaluate import evaluate_all
import matplotlib.pyplot as plt
%matplotlib inline

results = evaluate_all(models, X_test, y_test, feature_names=list(X_test.columns))

## 5. Kết luận \& Trực quan hóa

Các biểu đồ kết quả trực quan được xuất ra thư mục `outputs/`:

### So sánh Metrics
![](outputs/metrics_comparison.png)

### Confusion Matrices
![](outputs/confusion_matrices.png)

### ROC Curves
![](outputs/roc_curves.png)

### Feature Importance (Random Forest)
![](outputs/feature_importance.png)

- **Random Forest** cho kết quả tốt nhất khi có dữ liệu gán nhãn, nhận thức rõ nhất các đặc trưng `bytes_sent` và `upload_download_ratio`.
- Mô hình phát hiện rò rỉ dữ liệu nhận biệt rõ được hành vi truyền dữ liệu trái phép thông qua các đặc trưng ngữ cảnh (cổng giao tiếp, giờ giấc, tỉ lệ truy xuất ngoài).